In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4938,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-06      0  6.45  6.58  6.33   6.49  8.520715e+08   
            2018-07-09      1  6.51  6.76  6.50   6.76  1.255008e+09   
            2018-07-10      2  6.76  6.76  6.66   6.73  8.030390e+08   
            2018-07-11      3  6.56  6.61  6.50   6.58  7.447658e+08   
            2018-07-12      4  6.54  6.82  6.52   6.75  1.008659e+09   

                             volume  
code        date                     
000001.XSHE 2018-07-06  131957580.0  
            2018-07-09  188260099.0  
            2018-07-10  119750900.0  
            2018-07-11  113666923.0  
            2018-07-12  149984167.0

In [4]:
# 这里要计算乖离率，然后还要看看涨幅
dt['pre_close'] = dt['close'].shift(1) # 前一日收盘价
dt['up_percent'] = (dt['close']-dt['pre_close'])/dt['pre_close'] * 100 # 涨幅百分比形式
ma_timeperiod = [5,10,20]
for i in ma_timeperiod:
    dt.loc[:, f'MA{i}']=talib.MA(dt['close'], timeperiod=i) # 计算均线
    dt.loc[:, f'bias{i}']=(dt['close']-dt[f'MA{i}'])/dt[f'MA{i}'] *100 # 计算乖离率
dt.tail()

index   open   high    low  close         money  \
code        date                                                          
000001.XSHE 2026-09-23   1995  11.43  11.46  11.30  11.35  1.052015e+09   
            2026-09-24   1996  11.35  11.47  11.29  11.30  1.186737e+09   
            2026-09-28   1997  11.28  11.41  11.27  11.30  8.105758e+08   
            2026-09-29   1998  11.30  11.41  11.28  11.35  7.846328e+08   
            2026-09-30   1999  11.36  11.65  11.33  11.57  1.205815e+09   

                             volume  pre_close  up_percent     MA5    MA10  \
code        date                                                             
000001.XSHE 2026-09-23   92669758.0      11.46   -0.959860  11.420  11.479   
            2026-09-24  104381872.0      11.35   -0.440529  11.408  11.450   
            2026-09-28   71534143.0      11.30    0.000000  11.378  11.431   
            2026-09-29   69097909.0      11.30    0.442478  11.352  11.407   
            2026-09-30  104535745.0      11.35    1.938326  11.374  11.407   

                           MA20     bias5    bias10    bias20  
code        date                                               
000001.XSHE 2026-09-23  11.4995 -0.612960 -1.123791 -1.300057  
            2026-09-24  11.4975 -0.946704 -1.310044 -1.717765  
            2026-09-28  11.4925 -0.685533 -1.146006 -1.675005  
            2026-09-29  11.4865 -0.017618 -0.499693 -1.188352  
            2026-09-30  11.4820  1.723228  1.428947  0.766417

In [6]:
# 我这里先看一下相关系数吧
_columns = ['up_percent']
_columns.extend([f'bias{i}' for i in  ma_timeperiod])
dt2 = dt.loc[:, _columns]
dt2.corr() # 相关系数

,up_percent,bias5,bias10,bias20
up_percent,1.000000,0.735395,0.532967,0.376803
bias5,0.735395,1.000000,0.863889,0.650010
bias10,0.532967,0.863889,1.000000,0.868205
bias20,0.376803,0.650010,0.868205,1.000000


看情况是很少相关。